In [ ]:
import os, zipfile, glob, urllib.request
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

RAW = "/content/EduPredict/data/raw"
PROC = "/content/EduPredict/data/processed"
os.makedirs(RAW, exist_ok=True)
os.makedirs(PROC, exist_ok=True)

URLS = {
    "dropout": "https://archive.ics.uci.edu/static/public/697/predict+students+dropout+and+academic+success.zip",
    "performance": "https://archive.ics.uci.edu/static/public/320/student+performance.zip",
}

def download_and_extract(name, url):
    zip_path = f"{RAW}/{name}.zip"
    out_dir = f"{RAW}/{name}"
    if not os.path.exists(zip_path):
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=60) as r, open(zip_path, "wb") as f:
            f.write(r.read())
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(out_dir)
    # the performance zip contains another zip (student.zip) inside it
    for inner in glob.glob(f"{out_dir}/**/*.zip", recursive=True):
        with zipfile.ZipFile(inner) as z:
            z.extractall(os.path.dirname(inner))

def find_csv(filename):
    hits = glob.glob(f"{RAW}/**/{filename}", recursive=True)
    if not hits:
        raise FileNotFoundError(f"{filename} not found under {RAW}")
    return hits[0]

for name, url in URLS.items():
    try:
        download_and_extract(name, url)
        print(f"[OK] {name}")
    except Exception as e:
        print(f"[FAILED] {name}: {e}")

df_dropout = pd.read_csv(find_csv("data.csv"), sep=";")
df_math = pd.read_csv(find_csv("student-mat.csv"), sep=";")
df_por = pd.read_csv(find_csv("student-por.csv"), sep=";")
print(df_dropout.shape, df_math.shape, df_por.shape)

[OK] dropout
[OK] performance
(4424, 37) (395, 33) (649, 33)


In [ ]:
import os, glob, zipfile, urllib.request
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

RAW = "/content/EduPredict/data/raw"
PROC = "/content/EduPredict/data/processed"
os.makedirs(RAW, exist_ok=True)
os.makedirs(PROC, exist_ok=True)

URLS = {
    "dropout": "https://archive.ics.uci.edu/static/public/697/predict+students+dropout+and+academic+success.zip",
    "performance": "https://archive.ics.uci.edu/static/public/320/student+performance.zip",
}

def extract_all(zip_path, out_dir):
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(out_dir)
    # the performance zip contains another zip (student.zip) inside it
    for inner in glob.glob(f"{out_dir}/**/*.zip", recursive=True):
        with zipfile.ZipFile(inner) as z:
            z.extractall(os.path.dirname(inner))

def find_csv(name):
    hits = glob.glob(f"/content/**/{name}", recursive=True)
    return hits[0] if hits else None

# Step 1: if CSVs are missing, extract any zips already in Colab, then download
if not (find_csv("data.csv") and find_csv("student-mat.csv") and find_csv("student-por.csv")):
    for z in glob.glob("/content/**/*.zip", recursive=True):
        try:
            extract_all(z, f"{RAW}/from_{os.path.basename(z)[:-4]}")
        except Exception:
            pass

if not (find_csv("data.csv") and find_csv("student-mat.csv") and find_csv("student-por.csv")):
    for name, url in URLS.items():
        zip_path = f"{RAW}/{name}.zip"
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=60) as r, open(zip_path, "wb") as f:
                f.write(r.read())
            extract_all(zip_path, f"{RAW}/{name}")
            print(f"[OK] downloaded {name}")
        except Exception as e:
            print(f"[FAILED] {name}: {e}")

paths = {n: find_csv(n) for n in ["data.csv", "student-mat.csv", "student-por.csv"]}
print(paths)
missing = [n for n, p in paths.items() if p is None]
if missing:
    raise FileNotFoundError(f"Still missing: {missing}. See the fallback below.")

df_dropout = pd.read_csv(paths["data.csv"], sep=";")
df_math = pd.read_csv(paths["student-mat.csv"], sep=";")
df_por = pd.read_csv(paths["student-por.csv"], sep=";")
print(df_dropout.shape, df_math.shape, df_por.shape)

{'data.csv': '/content/EduPredict/data/raw/dropout/data.csv', 'student-mat.csv': '/content/EduPredict/data/raw/performance/student-mat.csv', 'student-por.csv': '/content/EduPredict/data/raw/performance/student-por.csv'}
(4424, 37) (395, 33) (649, 33)


In [ ]:
import glob
for p in glob.glob("/content/**/*", recursive=True):
    if p.endswith((".zip", ".csv")) and "sample_data" not in p:
        print(p)

/content/EduPredict/data/raw/performance.zip
/content/EduPredict/data/raw/dropout.zip
/content/EduPredict/data/raw/performance/student.zip
/content/EduPredict/data/raw/performance/student-mat.csv
/content/EduPredict/data/raw/performance/student-por.csv
/content/EduPredict/data/raw/dropout/data.csv


In [ ]:
df_dropout.columns = (
    df_dropout.columns.str.strip()
    .str.replace("Nacionality", "Nationality", regex=False)
)
print(df_dropout.columns[[4, 7]].tolist())

['Daytime/evening attendance', 'Nationality']


In [ ]:
datasets = {"dropout": df_dropout, "math": df_math, "portuguese": df_por}

for name, d in datasets.items():
    miss = d.isnull().sum()
    miss = miss[miss > 0]
    text_cols = d.select_dtypes(exclude="number").columns
    blanks = int(sum((d[c].astype(str).str.strip() == "").sum() for c in text_cols))
    print(f"{name:11s} | columns with missing: {len(miss)} | duplicates: {d.duplicated().sum()} | blank strings: {blanks}")

n_before = len(df_dropout)
df_dropout = df_dropout.drop_duplicates().reset_index(drop=True)
df_math = df_math.drop_duplicates().reset_index(drop=True)
df_por = df_por.drop_duplicates().reset_index(drop=True)
print("Dropout rows removed as duplicates:", n_before - len(df_dropout))

dropout     | columns with missing: 0 | duplicates: 0 | blank strings: 0
math        | columns with missing: 0 | duplicates: 0 | blank strings: 0
portuguese  | columns with missing: 0 | duplicates: 0 | blank strings: 0
Dropout rows removed as duplicates: 0


In [ ]:
dropout_categorical = [
    "Marital status", "Application mode", "Application order", "Course",
    "Previous qualification", "Nationality", "Mother's qualification",
    "Father's qualification", "Mother's occupation", "Father's occupation",
]
dropout_binary = [
    "Daytime/evening attendance", "Displaced", "Educational special needs",
    "Debtor", "Tuition fees up to date", "Gender", "Scholarship holder", "International",
]

for c in dropout_binary:
    assert set(df_dropout[c].unique()) <= {0, 1}, f"{c} has values other than 0/1"
for c in dropout_categorical + dropout_binary:
    df_dropout[c] = df_dropout[c].astype("int64")

print(df_dropout.dtypes.value_counts())

ordinal_cols = ["Medu", "Fedu", "traveltime", "studytime", "failures",
                "famrel", "freetime", "goout", "Dalc", "Walc", "health"]
for d in (df_math, df_por):
    for c in d.select_dtypes(exclude="number").columns:
        d[c] = d[c].astype(str).str.strip().str.lower()
    for c in ordinal_cols:
        d[c] = d[c].astype("int64")

print(sorted(df_math["Mjob"].unique()), sorted(df_math["sex"].unique()))

int64      29
float64     7
object      1
Name: count, dtype: int64
['at_home', 'health', 'other', 'services', 'teacher'] ['f', 'm']


In [ ]:
def iqr_report(d, cols):
    rows = []
    for c in cols:
        q1, q3 = d[c].quantile([0.25, 0.75])
        iqr = q3 - q1
        lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        n = int(((d[c] < lo) | (d[c] > hi)).sum())
        rows.append({"column": c, "min": d[c].min(), "max": d[c].max(),
                     "outliers": n, "pct": round(100 * n / len(d), 1)})
    return pd.DataFrame(rows).sort_values("outliers", ascending=False)

dropout_numeric = ["Previous qualification (grade)", "Admission grade", "Age at enrollment",
                   "Curricular units 1st sem (grade)", "Curricular units 2nd sem (grade)",
                   "Unemployment rate", "Inflation rate", "GDP"]

print("DROPOUT")
print(iqr_report(df_dropout, dropout_numeric).to_string(index=False))
print("\nMATH")
print(iqr_report(df_math, ["age", "absences", "G1", "G2", "G3"]).to_string(index=False))
print("\nPORTUGUESE")
print(iqr_report(df_por, ["age", "absences", "G1", "G2", "G3"]).to_string(index=False))

DROPOUT
                          column   min        max  outliers  pct
Curricular units 2nd sem (grade)  0.00  18.571429       877 19.8
Curricular units 1st sem (grade)  0.00  18.875000       726 16.4
               Age at enrollment 17.00  70.000000       441 10.0
  Previous qualification (grade) 95.00 190.000000       179  4.0
                 Admission grade 95.00 190.000000        86  1.9
               Unemployment rate  7.60  16.200000         0  0.0
                  Inflation rate -0.80   3.700000         0  0.0
                             GDP -4.06   3.510000         0  0.0

MATH
  column  min  max  outliers  pct
absences    0   75        15  3.8
      G2    0   19        13  3.3
     age   15   22         1  0.3
      G1    3   19         0  0.0
      G3    0   20         0  0.0

PORTUGUESE
  column  min  max  outliers  pct
      G2    0   19        25  3.9
absences    0   32        21  3.2
      G1    0   19        16  2.5
      G3    0   19        16  2.5
     age   15  

In [ ]:
df_dropout["Target"] = df_dropout["Target"].astype(str).str.strip().str.capitalize()
valid_targets = {"Dropout", "Enrolled", "Graduate"}
assert set(df_dropout["Target"].unique()) == valid_targets, "Unexpected Target labels!"
print(df_dropout["Target"].value_counts())

for d in (df_math, df_por):
    d["G3_zero_flag"] = (d["G3"] == 0).astype(int)

print("\nG3=0 students -> Math:", df_math["G3_zero_flag"].sum(),
      "| Portuguese:", df_por["G3_zero_flag"].sum())
zero = df_math[df_math["G3"] == 0]
print("Math G3=0: with G2>0:", (zero["G2"] > 0).sum(),
      "| with absences==0:", (zero["absences"] == 0).sum(), "of", len(zero))

Target
Graduate    2209
Dropout     1421
Enrolled     794
Name: count, dtype: int64

G3=0 students -> Math: 38 | Portuguese: 15
Math G3=0: with G2>0: 25 | with absences==0: 38 of 38


In [ ]:
def validate_dropout(d):
    checks = {}
    checks["no missing values"] = d.isnull().sum().sum() == 0
    checks["no duplicates"] = d.duplicated().sum() == 0
    checks["binary columns only 0/1"] = all(set(d[c].unique()) <= {0, 1} for c in dropout_binary)
    checks["admission grade in 0-200"] = d["Admission grade"].between(0, 200).all()
    checks["previous grade in 0-200"] = d["Previous qualification (grade)"].between(0, 200).all()
    checks["age 15-100"] = d["Age at enrollment"].between(15, 100).all()
    for s in ("1st", "2nd"):
        checks[f"{s} sem grade in 0-20"] = d[f"Curricular units {s} sem (grade)"].between(0, 20).all()
        checks[f"{s} sem approved <= enrolled"] = (
            d[f"Curricular units {s} sem (approved)"] <= d[f"Curricular units {s} sem (enrolled)"]
        ).all()
    checks["target labels valid"] = set(d["Target"].unique()) == valid_targets
    return checks

def validate_performance(d):
    checks = {}
    checks["no missing values"] = d.isnull().sum().sum() == 0
    checks["no duplicates"] = d.drop(columns="G3_zero_flag").duplicated().sum() == 0
    for g in ("G1", "G2", "G3"):
        checks[f"{g} in 0-20"] = d[g].between(0, 20).all()
    checks["absences >= 0"] = (d["absences"] >= 0).all()
    checks["age 10-30"] = d["age"].between(10, 30).all()
    return checks

for name, result in [("dropout", validate_dropout(df_dropout)),
                     ("math", validate_performance(df_math)),
                     ("portuguese", validate_performance(df_por))]:
    failed = {k: v for k, v in result.items() if not v}
    print(name, "-> ALL CHECKS PASSED" if not failed else f"FAILED: {failed}")

dropout -> ALL CHECKS PASSED
math -> ALL CHECKS PASSED
portuguese -> ALL CHECKS PASSED


In [ ]:
df_dropout.to_csv(f"{PROC}/dropout_clean.csv", index=False)
df_math.to_csv(f"{PROC}/student_math_clean.csv", index=False)
df_por.to_csv(f"{PROC}/student_por_clean.csv", index=False)

print(pd.read_csv(f"{PROC}/dropout_clean.csv").shape)
print(pd.read_csv(f"{PROC}/student_math_clean.csv").shape)
print(pd.read_csv(f"{PROC}/student_por_clean.csv").shape)

(4424, 37)
(395, 34)
(649, 34)
